# Transformer Model
### Weather and Water Temp Only Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import joblib
import matplotlib.dates as mdates
from sklearn.metrics import  r2_score
import tensorflow as tf
# set up relative imports
project_folder = Path.cwd().parent.parent
sys.path.append(str(project_folder))

In [2]:
from modeling.model.transformer import Transformer, CustomSchedule, get_callbacks
from modeling.utilities.data_prep import setup_sequential_data, create_split_dfs

#### Read in the Data

In [3]:
data_folder = Path(r'..\data\model-ready\water-weather')

# train
train_X_df = pd.read_parquet(data_folder / 'scaled-X-train.parquet')
train_y_df = pd.read_parquet(data_folder / 'scaled-Y-train.parquet') # the Y data is not actually scaled inspite of the name

# test
test_X_df = pd.read_parquet(data_folder / 'scaled-X-test.parquet')
test_y_df = pd.read_parquet(data_folder / 'scaled-Y-test.parquet') # the Y data is not actually scaled inspite of the name

# val
val_X_df = pd.read_parquet(data_folder / 'scaled-X-val.parquet')
val_y_df = pd.read_parquet(data_folder / 'scaled-Y-val.parquet') # the Y data is not actually scaled inspite of the name

scaler = joblib.load(data_folder / 'minmax_scaler.joblib')


#### Set up sequential data

In [4]:
# train
train_X_df.reset_index(drop=True, inplace=True)
train_y_df.reset_index(drop=True, inplace=True)

# test
test_X_df.reset_index(drop=True, inplace=True)
test_y_df.reset_index(drop=True, inplace=True)

# val
val_X_df.reset_index(drop=True, inplace=True)
val_y_df.reset_index(drop=True, inplace=True)

In [5]:
seq_size = 10
X_train, y_train, y_idx_train = setup_sequential_data(train_X_df, train_y_df, seq_size)
X_val, y_val, y_idx_val =  setup_sequential_data(val_X_df, val_y_df, seq_size)
X_test, y_test, y_idx_test =  setup_sequential_data(test_X_df, test_y_df, seq_size)

##### turn the dataframes into keras datasets

In [6]:
BATCH_SIZE = 32

train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
)

### Set up the transformer

In [7]:
num_layers = 4
d_model = 32
dff = 64
num_heads = 4
dropout_rate = 0.1

transformer = Transformer(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    dropout_rate=dropout_rate
    )

In [10]:
dummy = tf.zeros((1, X_train.shape[-2], X_train.shape[-1]))
transformer(dummy)
transformer.summary()

Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Encoder)               │ ?                      │        85,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Decoder)               │ ?                      │       152,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (1, 1)                 │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 237,537 (927.88 KB)

 Trainable params: 237,537 (927.88 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
learning_rate = CustomSchedule(d_model) # this thing is overkill imo

optimizer = tf.keras.optimizers.Adam(learning_rate,
                                     beta_1=0.9,
                                     beta_2=0.98,
                                     epsilon=1e-9
                                     )

In [12]:

transformer.compile(
    loss='mse',
    #optimizer=optimizer,
    optimizer='adam',
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(),
        ]
    )

In [13]:
history = transformer.fit(train_dataset,
                          epochs=1,
                          validation_data=val_dataset,
                          callbacks=get_callbacks()
                )

c:\Users\speco\anaconda3\envs\ml_capstone\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


691/691 ━━━━━━━━━━━━━━━━━━━━ 123s 114ms/step - loss: 14.1670 - mean_absolute_error: 2.4165 - val_loss: 0.6786 - val_mean_absolute_error: 0.5819 - learning_rate: 0.0010


### Model Evalutation

In [ ]:
y_pred = transformer.predict(test_dataset)


In [ ]:
test_x_unscaled = scaler.inverse_transform(test_X_df)
test_x_df = pd.DataFrame(columns=test_X_df.columns, data=test_x_unscaled)

In [ ]:
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred.flatten()
}, index=y_idx_test)

In [ ]:
test_x_df = test_x_df.join(results_df,
                how='inner')

In [ ]:
test_x_df['date'] = pd.to_datetime(test_x_df[['year','month','day']].astype(int), errors='coerce')

In [ ]:
test_x_df['errors'] = test_x_df.predicted - test_x_df.actual

In [ ]:
test_x_df.head(2)

In [ ]:
plt.hist(test_x_df.errors, bins='auto')
plt.title('Transformer Test Error')
plt.xlabel('Error (\u00b0C)', fontsize=16)
plt.show()

In [ ]:
pd.DataFrame(test_x_df.errors.abs().describe()).T.round(3)

In [ ]:
plot_df = test_x_df[:30].copy()
fig, ax = plt.subplots(figsize=(10,5))

ax.plot(plot_df.date, 
        plot_df.actual,
        label='True', 
        color='cornflowerblue'
        )
ax.plot(plot_df.date, 
        plot_df.predicted, 
        label='Predicted', 
        color='firebrick'
        )
ax.set_ylabel('Temperature (\u00b0C)', 
              fontsize=16
              )
ax.set_xlabel('Date', 
              fontsize=16
              )
ax.set_title('Transformer | Example Prediction', fontsize=18)
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.tick_params(axis='x', labelrotation=45)
fig.text(.9, -.05, f'{int(plot_df.year.unique()[0])}')
plt.show()

In [ ]:
mse = np.mean([x**2 for x in test_x_df.errors])
rmse = np.sqrt(mse)
r_2 = r2_score(y_test, y_pred)

print(f'RMSE = {round(rmse, 3)}')
print(f'r2 score = {round(r_2, 3)}')

In [ ]:
import os
os.getcwd()

In [14]:
# save the model
transformer.save('saved-models/transformer-1.keras')

In [17]:


from tensorflow import keras
from modeling.model.transformer import (
    Transformer, Encoder, Decoder, EncoderLayer, DecoderLayer,
    PositionalEmbedding, BaseAttention, CrossAttention,
    GlobalSelfAttention, CausalSelfAttention, FeedForward
)

model = keras.models.load_model(
    "saved-models/transformer-1.keras",
    custom_objects={
        "Transformer": Transformer,
        "Encoder": Encoder,
        "Decoder": Decoder,
        "EncoderLayer": EncoderLayer,
        "DecoderLayer": DecoderLayer,
        "PositionalEmbedding": PositionalEmbedding,
        "BaseAttention": BaseAttention,
        "CrossAttention": CrossAttention,
        "GlobalSelfAttention": GlobalSelfAttention,
        "CausalSelfAttention": CausalSelfAttention,
        "FeedForward": FeedForward,
    }
)

In [18]:
y_pred = model.predict(test_dataset)

148/148 ━━━━━━━━━━━━━━━━━━━━ 40s 195ms/step


In [ ]:
lin